# CLAUD·IA — Première traduction vocale

Ce notebook utilise SeamlessM4T v2 **avant fine-tuning**. Il reçoit un extrait audio, traduit la phrase et produit un nouvel audio. Nous mesurerons le temps de génération pour établir notre point de départ.

Premier essai proposé. Enregistrez « Gracias por venir » dans un fichier WAV court. La traduction anglaise de référence est « Thank you for coming. »

[Documentation de SeamlessM4T v2](https://huggingface.co/docs/transformers/model_doc/seamless_m4t_v2)

## 1. Installer les dépendances

Exécutez cette cellule lors de la première utilisation, puis redémarrez le noyau avant de poursuivre. Si vous avez utilisé une version précédente du notebook, cette cellule réinstalle les versions compatibles de Transformers et de Hugging Face Hub.

In [ ]:
%pip install "torch>=2.6" "transformers[sentencepiece]==5.22.1" "huggingface-hub>=1.32,<3" soundfile scipy

## 2. Choisir l'audio et la langue cible

Dans Jupyter, placez votre fichier WAV à côté du notebook. Dans Colab, ajoutez-le depuis le panneau **Fichiers**. Indiquez son nom ci-dessous et exécutez la cellule pour écouter l'original.

Pour le premier essai, laissez `tgt_lang = "eng"`. Les autres choix sont `fra` et `spa`.

In [ ]:
from math import gcd
from pathlib import Path

import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import resample_poly

audio_path = Path("gracias_por_venir.wav")
tgt_lang = "eng"
input_rate = 16_000

if not audio_path.is_file():
    raise FileNotFoundError(f"Ajoutez le fichier WAV {audio_path.name} dans le dossier du notebook.")

audio, original_rate = sf.read(audio_path, dtype="float32", always_2d=True)
if len(audio) == 0:
    raise ValueError("Le fichier audio est vide.")

# Une moyenne des canaux donne un signal mono.
mono = audio.mean(axis=1)
if original_rate != input_rate:
    divisor = gcd(original_rate, input_rate)
    mono = resample_poly(mono, input_rate // divisor, original_rate // divisor)

print(f"Audio original • {len(mono) / input_rate:.1f} s • mono à {input_rate} Hz")
display(Audio(mono, rate=input_rate))

## 3. Charger le Transformer

Le premier chargement télécharge plusieurs gigaoctets de paramètres. Lancez cette cellule une seule fois par session. Le CPU est sélectionné par défaut. Dans Colab avec un environnement GPU, vous pouvez passer `use_gpu` à `True`.

In [ ]:
import torch
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToSpeech

model_id = "facebook/seamless-m4t-v2-large"
use_gpu = False

if use_gpu and not torch.cuda.is_available():
    raise RuntimeError("Activez un environnement GPU ou laissez use_gpu à False.")

device = torch.device("cuda" if use_gpu else "cpu")
dtype = torch.float16 if use_gpu else torch.float32

processor = AutoProcessor.from_pretrained(model_id)
model = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
    model_id, dtype=dtype, use_safetensors=True
).to(device)
model.eval()
print(f"Modèle chargé sur {device}.")

## 4. Traduire et écouter

Le modèle produit une traduction textuelle intermédiaire et génère l'audio à partir de celle-ci. Le texte affiché aide à vérifier la traduction pendant le travail pratique. La sortie à écouter se trouve juste en dessous.

In [ ]:
from time import perf_counter

if tgt_lang not in {"eng", "fra", "spa"}:
    raise ValueError("Choisissez eng, fra ou spa comme langue cible.")

inputs = processor(audio=mono, sampling_rate=input_rate, return_tensors="pt").to(device)
inputs["input_features"] = inputs["input_features"].to(dtype=dtype)

if use_gpu:
    torch.cuda.synchronize()
start = perf_counter()
with torch.inference_mode():
    result = model.generate(**inputs, tgt_lang=tgt_lang, return_intermediate_token_ids=True)
if use_gpu:
    torch.cuda.synchronize()
elapsed = perf_counter() - start

translation = processor.batch_decode(result.sequences.cpu(), skip_special_tokens=True)[0]
length = result.waveform_lengths[0].item()
translated_audio = result.waveform[0, :length].float().cpu().numpy()

print(f"Traduction générée ({tgt_lang}) — {translation}")
print(f"Temps de génération — {elapsed:.2f} s, hors chargement du modèle")
display(Audio(translated_audio, rate=model.config.sampling_rate))